In [2]:
import multiprocessing
print(multiprocessing.cpu_count())

12


In [4]:
!pip install -q rank_bm25

In [6]:
!python -m spacy download fr_core_news_sm
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 142.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
from rank_bm25 import BM25Okapi
import pickle
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

print("Loading datasets...")
lang = "en"

collection_data = load_dataset(
    "sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims",
    "collection",
    token="<HF_TOKEN>"
)["collection"]

train_data = load_dataset(
    "sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims",
    lang,
    token="<HF_TOKEN>"
)["train"]

# =========================
# 1. Load spaCy (FAST MODE)
# =========================
print("Loading NLP model...")
if lang == "fr":
    nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner", "tagger"])
elif lang == "en":
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "tagger"])
elif lang == "de":
    nlp = spacy.load("de_core_news_sm", disable=["parser", "ner", "tagger"])

def tokenize_doc(doc):
    return [t.lemma_ for t in doc if t.is_alpha and not t.is_stop]

# =========================
# 2. CLEAN COLLECTION (BATCHED)
# =========================
print("Cleaning collection with spaCy.pipe...")

texts = [f"{item['title']} {item['abstract']}" for item in collection_data]
doc_ids = [item['pubkey'] for item in collection_data]

tokenized_corpus = []
for doc in tqdm(nlp.pipe(texts, batch_size=64), total=len(texts)):
    tokenized_corpus.append(tokenize_doc(doc))

# =========================
# 3. BUILD BM25
# =========================
print("Building BM25 index...")
bm25 = BM25Okapi(tokenized_corpus)

# =========================
# 4. CLEAN QUERIES (BATCHED)
# =========================
print("Processing queries with spaCy.pipe...")

queries = [item['text'] for item in train_data]
processed_queries = list(nlp.pipe(queries, batch_size=64))

query_tokens = [tokenize_doc(doc) for doc in processed_queries]

# =========================
# 5. MULTIPROCESS HARD NEGATIVE MINING
# =========================
print("Mining hard negatives with multiprocessing...")

def process_one(args):
    i, item = args
    tokenized_query = query_tokens[i]

    doc_scores = bm25.get_scores(tokenized_query)

    # FAST top-k instead of full sort
    top_doc_indices = np.argpartition(doc_scores, -10)[-10:]
    top_doc_indices = top_doc_indices[
        np.argsort(doc_scores[top_doc_indices])[::-1]
    ]

    for idx in top_doc_indices:
        candidate_pubkey = doc_ids[idx]
        if candidate_pubkey != item['pubkey']:
            return (item['index'], [candidate_pubkey])

    return (item['index'], [])

# Multiprocessing
num_workers = 8

with Pool(num_workers) as pool:
    results = list(
        tqdm(
            pool.imap(process_one, enumerate(train_data)),
            total=len(train_data)
        )
    )

# جمع results
hard_negatives = {k: v for k, v in results if v}

# =========================
# 6. SAVE
# =========================
output_file = f"hard_negs_{lang}.pkl"

with open(output_file, "wb") as f:
    pickle.dump(hard_negatives, f)

print(f"Success! Saved hard negatives to {output_file}")

Loading datasets...
Loading French NLP model...
Cleaning collection with spaCy.pipe...


100%|██████████| 10000/10000 [03:14<00:00, 51.37it/s]


Building BM25 index...
Processing queries with spaCy.pipe...
Mining hard negatives with multiprocessing...


100%|██████████| 2807/2807 [00:17<00:00, 157.32it/s]

Success! Saved hard negatives to hard_negs_fr.pkl
